## Hyperparameter Optimization - Grid Search


In this example we will use grid search to select the hyperparameter configuration for the Basket option example using <code>Optuna</code>

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.nn.init as init
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import trange
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from torch.optim import Adam, RMSprop, Adagrad
import optuna

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
print("Device:", device)

### Read Basket Option Data

In [ ]:
base_filename = 'basket_option100000x10'

In [ ]:
# Basket data stored in 10 files - load and concatenate
basket_data_list = list()
for i in range(0,10):
    new_basket_data = pd.read_csv(base_filename + str(i) + '.csv', index_col=[0])
    basket_data_list.append(new_basket_data) 
    
basket_data = pd.concat(basket_data_list)
basket_data.reset_index(inplace=True)

In [ ]:
drop_list = ['index','errorEstimate', 'samples', 'processTime']
basket_dataset = basket_data.drop(drop_list, axis=1)
basket_dataset.head()

In [ ]:
train_df = basket_dataset[:-20000]
validate_df = basket_dataset[-20000:]

def prepareData(df):
    y = pd.DataFrame(df['optionValue'])
    X = df.drop(['optionValue'], axis=1)
    return X, y

X_train, y_train = prepareData(train_df)
X_val, y_val = prepareData(validate_df)

### Standard Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler
standard_scalar = StandardScaler()
X_train = standard_scalar.fit_transform(X_train)
X_val = standard_scalar.transform(X_val)

### Build Model

In [ ]:
def create_optimizer(name, learning_rate, parameters):
    if name == 'Adam':
        opt = Adam(parameters, lr=learning_rate)
    elif name == 'RMSprop':
        opt = RMSprop(parameters, lr=learning_rate)
    else:
        opt = Adagrad(parameters, lr=learning_rate)
    return opt

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, no_layers=1, no_neurons=1, input_shape=28):
        super(FeedForward, self).__init__()
        self.layers = nn.ModuleList()
        
        # Creating the first layer that takes the input
        prev_size = input_shape
        size = no_neurons
        for _ in range(no_layers):
            self.layers.append(nn.Linear(prev_size, size))
            prev_size = size
        self.output_layer = nn.Linear(prev_size, 1)

    def forward(self, x):
        for layer in self.layers:
            x = F.relu(layer(x))
        x = self.output_layer(x)
        return x

In [ ]:
no_epochs = 30
batch_size = 2048

In [ ]:
train_x = torch.Tensor(X_train).to(device)
train_y = torch.Tensor(y_train.to_numpy().copy()).to(device)
train_dataset = TensorDataset(train_x, train_y)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, 
                                           shuffle=True, drop_last=True)

val_x = torch.Tensor(X_val).to(device)
val_y = torch.Tensor(y_val.to_numpy().copy()).to(device)
val_dataset = TensorDataset(val_x, val_y)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, 
                                          shuffle=False, drop_last=True)

### Construct Grid Search Tuner

In [ ]:
def train_model(model, train_loader, test_loader, loss_fn, optimizer, epochs):
    train_errors = []
    test_errors = []

    tqdm_epoch = trange(epochs)
    for epoch in tqdm_epoch:
        model.train()
        train_loss = 0.0

        # Training
        for batch_X, batch_y in train_loader:
            # Forward pass
            outputs = model(batch_X)
            loss = loss_fn(outputs.squeeze(), batch_y.squeeze())
            
            # Backward pass and optimization
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * batch_X.size(0)

        train_loss /= len(train_loader.dataset)
        train_errors.append(train_loss)
        
        # Evaluation on test set
        model.eval()
        test_loss = 0.0
        with torch.no_grad():
            for batch_X, batch_y in test_loader:
                outputs = model(batch_X)
                loss = loss_fn(outputs.squeeze(), batch_y.squeeze())
                test_loss += loss.item() * batch_X.size(0)

        test_loss /= len(test_loader.dataset)
        test_errors.append(test_loss)

        tqdm_epoch.set_description(f"Epoch {epoch+1}/{epochs} \
            - Train loss: {train_loss:.4f}, Test loss: {test_loss:.4f}")

    history = dict()
    history['train_loss'] = train_errors
    history['test_loss'] = test_errors
    return history

In [ ]:
def objective(trial):
    # Define the hyperparameters
    no_layers = trial.suggest_int('no_layers', 1, 6)
    layer_width = trial.suggest_int('hidden_size', 32, 512, step=32)
    optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'RMSprop', 'AdaGrad'])
    lr = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
    print(f"no_layers: {no_layers}, layer_width: {layer_width}, optimizer_name: {optimizer_name}, lr: {lr}")
    model = FeedForward(no_layers, layer_width).to(device)
    optimizer = create_optimizer(optimizer_name, lr, model.parameters())
    loss_fn = nn.MSELoss()
    history = train_model(model, train_loader, val_loader, loss_fn, optimizer, no_epochs)
    return history['test_loss'][-1]

### Run Search
We now run the search, passing in the training and validation data sets,

In [ ]:
param_grid = {
    'lr': [1e-4, 1e-3, 1e-2],
    'optimizer': ['Adam', 'RMSprop', 'AdaGrad'],
    'no_layers': [1, 3, 5],
    'hidden_size': [16, 64, 256]
}

In [ ]:
study = optuna.create_study(sampler=optuna.samplers.GridSampler(param_grid))
print(f"Sampler is {study.sampler.__class__.__name__}")

In [ ]:
study.optimize(objective, n_trials=3*3*3*3)

In [ ]:
import pickle

# Save the sampler with pickle to be loaded later.
with open("gridstudy.pkl", "wb") as fout:
    pickle.dump(study.sampler, fout)

In [ ]:
df = study.trials_dataframe(attrs=("number", "value", "params", "state"))

In [ ]:
min_index = df['value'].idxmin()
min_row = df.loc[min_index]
min_row

In [ ]:
df

In [ ]:
df.to_latex()